# Code LLM Forge - Complete Pipeline
> **Build, fine-tune, and benchmark your own coding LLM for code generation, tool calling, and reasoning.**

## Pipeline Overview
| Section | What It Does | Time Estimate |
|---------|-------------|---------------|
| **0. Setup** | Install deps, configure GPU | 2 min |
| **1. Data Prep** | Download & combine 5 datasets | 10-20 min |
| **2. Stage 1** | Code + Reasoning fine-tune (QLoRA) | 8-12 hrs (A100) |
| **3. Stage 2** | Tool Calling specialization | 4-6 hrs (A100) |
| **4. Stage 3** | DPO preference optimization | 2-3 hrs |
| **5. Benchmark** | 3-tier contamination-resistant eval | 1-2 hrs |
| **6. Export** | Convert to GGUF for llama.cpp | 15 min |

**GPU Required:** A100 40/80GB or H100 (Google Colab Pro)
**Base Model:** Qwen2.5-Coder-7B-Instruct (best coding benchmarks + native tool calling)
**Also supported:** Gemma 4 E2B/E4B (excellent tool calling, Apache 2.0)

---


## Section 0: Setup & Installation

In [ ]:
# ============================================================
# SECTION 0: SETUP
# ============================================================
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes
!pip install -q datasets wandb scipy numpy evalplus huggingface_hub

In [ ]:
import torch
import json
import os
import re
import random
import numpy as np
from scipy import stats
from datetime import datetime
from tqdm.auto import tqdm

# Seed everything
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# GPU info
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"CUDA: {torch.version.cuda}")

# Create directories
for d in ["prepared_data", "outputs/stage1", "outputs/stage2", "outputs/stage3_dpo",
          "benchmarks", "gguf_output"]:
    os.makedirs(d, exist_ok=True)
print("Directories ready")

In [ ]:
# ===== MODEL SELECTION =====
# Choose your base model. Both are excellent choices.

# Option A: Qwen2.5-Coder-7B-Instruct (DEFAULT - best coding benchmarks, native tool calling)
BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"

# Option B: Gemma 4 E4B (excellent tool calling, multimodal, Apache 2.0)
# BASE_MODEL = "google/gemma-4-E4B-it"

# Option C: Gemma 4 E2B (smallest, runs on 8GB VRAM, surprisingly good at tool calling)
# BASE_MODEL = "google/gemma-4-E2B-it"

MAX_SEQ_LENGTH = 4096
print(f"Base model: {BASE_MODEL}")

---
## Section 1: Data Preparation
> Download, format, and combine datasets for all training stages.

**Datasets used:**
- **CodeFeedback-Filtered-Instruction** (156K) - complexity-filtered code instructions
- **Magicoder-OSS-Instruct-75K** - real open-source grounded code
- **Magicoder-Evol-Instruct-110K** - evolved complexity tasks
- **NousResearch/hermes-function-calling-v1** - multi-turn tool calling
- **glaiveai/glaive-function-calling-v2** - 113K function calling examples


In [ ]:
# Download all datasets
print("Downloading Stage 1 datasets (Code + Reasoning)...")
from datasets import load_dataset

ds_codefeedback = load_dataset("m-a-p/CodeFeedback-Filtered-Instruction", split="train")
print(f"  CodeFeedback: {len(ds_codefeedback):,}")

ds_magicoder_oss = load_dataset("ise-uiuc/Magicoder-OSS-Instruct-75K", split="train")
print(f"  Magicoder-OSS: {len(ds_magicoder_oss):,}")

ds_magicoder_evol = load_dataset("ise-uiuc/Magicoder-Evol-Instruct-110K", split="train")
print(f"  Magicoder-Evol: {len(ds_magicoder_evol):,}")

print("\nDownloading Stage 2 datasets (Tool Calling)...")
ds_hermes_fc = load_dataset("NousResearch/hermes-function-calling-v1", split="train")
print(f"  Hermes FC: {len(ds_hermes_fc):,}")

ds_glaive = load_dataset("glaiveai/glaive-function-calling-v2", split="train")
print(f"  Glaive FC: {len(ds_glaive):,}")

print("\nAll datasets downloaded!")

In [ ]:
# Format all datasets into unified chat message format
SYSTEM_CODE = "You are a helpful coding assistant. Think step-by-step and provide clear, well-documented code."
SYSTEM_TOOL = """You are a helpful coding assistant with tool-calling capabilities.

# Tools
You may call one or more functions to assist with the user query.
For each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:
<tool_call>
{"name": <function-name>, "arguments": <args-json-object>}
</tool_call>"""

def format_codefeedback(ex):
    return {"messages": [
        {"role": "system", "content": SYSTEM_CODE},
        {"role": "user", "content": ex.get("query", ex.get("instruction", ""))},
        {"role": "assistant", "content": ex.get("answer", ex.get("response", ""))}
    ], "source": "codefeedback"}

def format_magicoder(ex):
    return {"messages": [
        {"role": "system", "content": SYSTEM_CODE},
        {"role": "user", "content": ex.get("problem", ex.get("instruction", ""))},
        {"role": "assistant", "content": ex.get("solution", ex.get("response", ""))}
    ], "source": "magicoder"}

def format_hermes_fc(ex):
    convos = ex.get("conversations", [])
    msgs = []
    role_map = {"system": "system", "human": "user", "gpt": "assistant", "tool": "tool"}
    for turn in convos:
        role = role_map.get(turn.get("from", ""), turn.get("from", ""))
        msgs.append({"role": role, "content": turn.get("value", "")})
    if not any(m["role"] == "system" for m in msgs):
        msgs.insert(0, {"role": "system", "content": SYSTEM_TOOL})
    return {"messages": msgs, "source": "hermes_fc"}

def format_glaive(ex):
    msgs = [{"role": "system", "content": ex.get("system", SYSTEM_TOOL)}]
    chat = ex.get("chat", "")
    parts = chat.split("USER: ")
    for part in parts[1:]:
        if "ASSISTANT: " in part:
            user_msg, asst_msg = part.split("ASSISTANT: ", 1)
            msgs.append({"role": "user", "content": user_msg.strip()})
            msgs.append({"role": "assistant", "content": asst_msg.strip()})
        elif part.strip():
            msgs.append({"role": "user", "content": part.strip()})
    return {"messages": msgs, "source": "glaive_fc"}

print("Formatting functions ready")

In [ ]:
# Process Stage 1 data
print("Formatting Stage 1 (Code + Reasoning)...")
stage1 = []
for ex in tqdm(ds_codefeedback, desc="CodeFeedback"):
    try:
        f = format_codefeedback(ex)
        if f["messages"][1]["content"] and f["messages"][2]["content"]:
            stage1.append(f)
    except: pass

for ex in tqdm(ds_magicoder_oss, desc="Magicoder-OSS"):
    try:
        f = format_magicoder(ex)
        if f["messages"][1]["content"] and f["messages"][2]["content"]:
            stage1.append(f)
    except: pass

for ex in tqdm(ds_magicoder_evol, desc="Magicoder-Evol"):
    try:
        f = format_magicoder(ex)
        if f["messages"][1]["content"] and f["messages"][2]["content"]:
            stage1.append(f)
    except: pass

random.shuffle(stage1)
with open("prepared_data/stage1.jsonl", "w") as fout:
    for s in stage1:
        fout.write(json.dumps(s) + "\n")
print(f"Stage 1: {len(stage1):,} samples saved")

# Process Stage 2 data
print("\nFormatting Stage 2 (Tool Calling)...")
stage2 = []
for ex in tqdm(ds_hermes_fc, desc="Hermes FC"):
    try:
        f = format_hermes_fc(ex)
        if len(f["messages"]) >= 2:
            stage2.append(f)
    except: pass

limit = int(len(stage2) * 0.42)
for i, ex in enumerate(tqdm(ds_glaive, desc="Glaive FC")):
    if i >= limit: break
    try:
        f = format_glaive(ex)
        if len(f["messages"]) >= 3:
            stage2.append(f)
    except: pass

random.shuffle(stage2)
with open("prepared_data/stage2.jsonl", "w") as fout:
    for s in stage2:
        fout.write(json.dumps(s) + "\n")
print(f"Stage 2: {len(stage2):,} samples saved")

print(f"\nTotal: {len(stage1) + len(stage2):,} samples ready for training!")

---
## Section 2: Stage 1 — Code + Reasoning Fine-Tuning (SFT)
> QLoRA fine-tune on ~340K code generation and reasoning examples.

**Method:** QLoRA (4-bit quantization + LoRA rank 32)
**Time:** ~8-12 hours on A100, ~5-8 hours on H100


In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# Load model
print(f"Loading {BASE_MODEL}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
# Load and format Stage 1 dataset
samples = []
with open("prepared_data/stage1.jsonl") as f:
    for line in f:
        samples.append(json.loads(line))
print(f"Loaded {len(samples):,} Stage 1 samples")

def apply_template(sample):
    text = tokenizer.apply_chat_template(
        sample["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset_s1 = Dataset.from_list(samples)
dataset_s1 = dataset_s1.map(apply_template, remove_columns=dataset_s1.column_names, num_proc=4)
print(f"Formatted: {len(dataset_s1):,} samples")

In [ ]:
# Train Stage 1
trainer_s1 = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=dataset_s1, dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH, dataset_num_proc=4, packing=True,
    args=TrainingArguments(
        output_dir="outputs/stage1",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        num_train_epochs=2,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        weight_decay=0.01,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25, save_strategy="steps", save_steps=500,
        save_total_limit=3, optim="adamw_8bit", seed=42, report_to="none",
    ),
)

print("Starting Stage 1 training...")
stats_s1 = trainer_s1.train()
print(f"Stage 1 done! Time: {stats_s1.metrics['train_runtime']/3600:.1f}h, Loss: {stats_s1.metrics['train_loss']:.4f}")

# Save & merge
model.save_pretrained("outputs/stage1/lora")
tokenizer.save_pretrained("outputs/stage1/lora")
merged = model.merge_and_unload()
merged.save_pretrained("outputs/stage1/merged")
tokenizer.save_pretrained("outputs/stage1/merged")
print("Stage 1 model saved!")

# Free memory
del model, merged, trainer_s1
torch.cuda.empty_cache()

---
## Section 3: Stage 2 — Tool Calling Specialization (SFT)
> Layer function-calling capabilities on top of the Stage 1 model.

**Key:** Uses lower learning rate (1e-4 vs 2e-4) and lower LoRA rank (16 vs 32) to preserve code skills.


In [ ]:
# Load Stage 1 model as base for Stage 2
S1_MODEL = "outputs/stage1/merged"
if not os.path.exists(S1_MODEL):
    print("Stage 1 model not found, using base model")
    S1_MODEL = BASE_MODEL

print(f"Loading {S1_MODEL}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=S1_MODEL, max_seq_length=MAX_SEQ_LENGTH, dtype=None, load_in_4bit=True)

model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16,  # Lower rank for focused task
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
print("Stage 2 LoRA adapters added")

In [ ]:
# Load Stage 2 dataset
samples2 = []
with open("prepared_data/stage2.jsonl") as f:
    for line in f:
        samples2.append(json.loads(line))

def apply_template_safe(sample):
    msgs = [m for m in sample["messages"] if m.get("content", "").strip()]
    try:
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    except:
        text = ""
        for msg in msgs:
            text += f"<|im_start|>{msg['role']}\n{msg['content']}<|im_end|>\n"
    return {"text": text}

dataset_s2 = Dataset.from_list(samples2)
dataset_s2 = dataset_s2.map(apply_template_safe, remove_columns=dataset_s2.column_names, num_proc=4)
dataset_s2 = dataset_s2.filter(lambda x: len(x["text"]) > 50)
print(f"Stage 2 dataset: {len(dataset_s2):,} samples")

In [ ]:
# Train Stage 2
trainer_s2 = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=dataset_s2, dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH, dataset_num_proc=4, packing=True,
    args=TrainingArguments(
        output_dir="outputs/stage2",
        per_device_train_batch_size=2, gradient_accumulation_steps=8,
        num_train_epochs=2,
        learning_rate=1e-4,  # Lower LR to preserve Stage 1 skills
        lr_scheduler_type="cosine", warmup_ratio=0.1,
        weight_decay=0.01,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25, save_strategy="steps", save_steps=500,
        save_total_limit=3, optim="adamw_8bit", seed=42, report_to="none",
    ),
)

print("Starting Stage 2 training...")
stats_s2 = trainer_s2.train()
print(f"Stage 2 done! Time: {stats_s2.metrics['train_runtime']/3600:.1f}h, Loss: {stats_s2.metrics['train_loss']:.4f}")

model.save_pretrained("outputs/stage2/lora")
tokenizer.save_pretrained("outputs/stage2/lora")
merged = model.merge_and_unload()
merged.save_pretrained("outputs/stage2/merged")
tokenizer.save_pretrained("outputs/stage2/merged")
print("Stage 2 model saved!")

del model, merged, trainer_s2
torch.cuda.empty_cache()

---
## Section 4: Stage 3 — DPO Preference Optimization (Optional)
> Polish the model by training it to prefer good responses over bad ones.

**Note:** This uses starter examples. For best results, scale to 1,000-10,000+ pairs
generated from your model's outputs scored by tests or a reward model.


In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
from trl import DPOTrainer, DPOConfig
PatchDPOTrainer()

S2_MODEL = "outputs/stage2/merged"
if not os.path.exists(S2_MODEL):
    S2_MODEL = BASE_MODEL
    print("Stage 2 not found, using base model")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=S2_MODEL, max_seq_length=2048, dtype=None, load_in_4bit=True)

model = FastLanguageModel.get_peft_model(
    model, r=8, lora_alpha=8,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0, bias="none",
    use_gradient_checkpointing="unsloth", random_state=42,
)
print("DPO model loaded")

In [ ]:
# DPO preference pairs - chosen (good) vs rejected (bad)
dpo_pairs = [
    {
        "prompt": "<|im_start|>user\nWrite a function to check if a string is a palindrome.<|im_end|>\n<|im_start|>assistant\n",
        "chosen": "def is_palindrome(s: str) -> bool:\n    cleaned = ''.join(c.lower() for c in s if c.isalnum())\n    return cleaned == cleaned[::-1]",
        "rejected": "def is_palindrome(s):\n    return s == s[::-1]  # BUG: no case/punctuation handling"
    },
    {
        "prompt": "<|im_start|>user\nWhat is the weather in Tokyo?<|im_end|>\n<|im_start|>assistant\n",
        "chosen": '<tool_call>\n{"name": "get_weather", "arguments": {"city": "Tokyo"}}\n</tool_call>',
        "rejected": "The weather in Tokyo is usually mild. (Made up without calling a tool)"
    },
    {
        "prompt": "<|im_start|>user\nExplain what a binary search tree is.<|im_end|>\n<|im_start|>assistant\n",
        "chosen": "A BST is a data structure where each node has at most two children. Left subtree values < node, right subtree values > node. This enables O(log n) search, insert, delete on average.",
        "rejected": '<tool_call>\n{"name": "search_web", "arguments": {"query": "BST"}}\n</tool_call>\nWrong: should answer directly.'
    },
    {
        "prompt": "<|im_start|>user\nFind the kth largest element in an array.<|im_end|>\n<|im_start|>assistant\n",
        "chosen": "Step by step:\n1. Sort approach: O(n log n)\n2. Min-heap of size k: O(n log k)\n3. Quickselect: O(n) average\n\nimport heapq\ndef kth_largest(nums, k):\n    return heapq.nlargest(k, nums)[-1]",
        "rejected": "def kth_largest(nums, k):\n    nums.sort()\n    return nums[-k]\n# No reasoning"
    },
]

dataset_dpo = Dataset.from_list(dpo_pairs)
print(f"DPO pairs: {len(dataset_dpo)} (scale to 1K-10K for production)")

In [ ]:
# Train DPO
dpo_trainer = DPOTrainer(
    model=model, ref_model=None, tokenizer=tokenizer,
    train_dataset=dataset_dpo,
    args=DPOConfig(
        output_dir="outputs/stage3_dpo",
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        num_train_epochs=3, learning_rate=5e-5,
        lr_scheduler_type="cosine", warmup_ratio=0.1,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=1, save_strategy="epoch",
        optim="adamw_8bit", seed=42, beta=0.1,
        max_length=2048, max_prompt_length=512, report_to="none",
    ),
)

stats_dpo = dpo_trainer.train()
print(f"DPO done! Time: {stats_dpo.metrics['train_runtime']/60:.1f} min")

# Save final model
merged = model.merge_and_unload()
merged.save_pretrained("outputs/stage3_dpo/merged")
tokenizer.save_pretrained("outputs/stage3_dpo/merged")
print("Final DPO model saved!")

del model, merged, dpo_trainer
torch.cuda.empty_cache()

---
## Section 5: Benchmarking Suite
> 3-tier contamination-resistant evaluation. No reward hacking.

| Tier | Tests | Trust |
|------|-------|-------|
| **Tier 1** | Custom tool-calling + reasoning benchmarks | Zero contamination |
| **Tier 2** | EvalPlus (HumanEval+, MBPP+) | Low contamination (extra test cases) |
| **Tier 3** | Standard HumanEval/MBPP (reference only) | Assume contaminated |


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Models to evaluate (only those that exist)
EVAL_MODELS = {}
for name, path in [
    ("base", BASE_MODEL),
    ("stage1", "outputs/stage1/merged"),
    ("stage2", "outputs/stage2/merged"),
    ("stage3_dpo", "outputs/stage3_dpo/merged"),
]:
    if os.path.exists(path) or "/" in path:
        EVAL_MODELS[name] = path
print(f"Models to evaluate: {list(EVAL_MODELS.keys())}")

In [ ]:
# Custom Tool-Calling Benchmark (Tier 1 - Zero Contamination)
TC_PROBLEMS = [
    {
        "id": "tc_001", "difficulty": "easy",
        "query": "What is the weather in Tokyo right now?",
        "functions": [
            {"name": "get_weather", "description": "Get weather for a city",
             "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}},
            {"name": "get_stock_price", "description": "Get stock price",
             "parameters": {"type": "object", "properties": {"ticker": {"type": "string"}}, "required": ["ticker"]}}
        ],
        "expected_fn": "get_weather", "expected_args": {"city": "Tokyo"},
    },
    {
        "id": "tc_002", "difficulty": "medium",
        "query": "Send an email to alice@test.com with subject Meeting and body See you at 3pm.",
        "functions": [
            {"name": "send_email", "description": "Send an email",
             "parameters": {"type": "object", "properties": {
                 "to": {"type": "string"}, "subject": {"type": "string"}, "body": {"type": "string"}
             }, "required": ["to", "subject", "body"]}}
        ],
        "expected_fn": "send_email", "expected_args": {"to": "alice@test.com", "subject": "Meeting"},
    },
    {
        "id": "tc_003", "difficulty": "hard",
        "query": "Explain how recursion works in Python.",
        "functions": [
            {"name": "run_code", "description": "Execute Python code",
             "parameters": {"type": "object", "properties": {"code": {"type": "string"}}, "required": ["code"]}}
        ],
        "should_call": False,  # Should NOT call any function
    },
]

with open("benchmarks/tool_calling_bench.json", "w") as f:
    json.dump(TC_PROBLEMS, f, indent=2)
print(f"Tool-calling benchmark: {len(TC_PROBLEMS)} problems")

In [ ]:
# Code Reasoning Benchmark (Tier 1)
REASON_PROBLEMS = [
    {
        "id": "r_001", "difficulty": "medium",
        "problem": "Write a function that finds the longest consecutive sequence in an unsorted list.\nExample: [100, 4, 200, 1, 3, 2] -> 4 (the sequence 1,2,3,4).\nThink step by step.",
        "fn_name": "longest_consecutive",
        "tests": [
            {"input": [100, 4, 200, 1, 3, 2], "expected": 4},
            {"input": [0, 3, 7, 2, 5, 8, 4, 6, 0, 1], "expected": 9},
            {"input": [], "expected": 0},
            {"input": [1], "expected": 1},
        ]
    },
    {
        "id": "r_002", "difficulty": "hard",
        "problem": "Write a function that checks if parentheses, brackets, and braces are balanced.\nExamples: '({[]})' -> True, '([)]' -> False\nThink step by step.",
        "fn_name": "is_balanced",
        "tests": [
            {"input": "({[]})", "expected": True},
            {"input": "([)]", "expected": False},
            {"input": "", "expected": True},
            {"input": "((", "expected": False},
        ]
    },
]

with open("benchmarks/reasoning_bench.json", "w") as f:
    json.dump(REASON_PROBLEMS, f, indent=2)
print(f"Reasoning benchmark: {len(REASON_PROBLEMS)} problems")

In [ ]:
def eval_tool_calling(model_path, problems):
    """Evaluate tool-calling capability."""
    print(f"  Loading {model_path.split('/')[-1]}...")
    tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
    mdl.eval()
    scores = []
    for p in problems:
        tools_json = json.dumps(p["functions"], indent=2)
        msgs = [
            {"role": "system", "content": f"You are a helpful assistant with tools.\n<tools>\n{tools_json}\n</tools>\nUse <tool_call> tags for function calls."},
            {"role": "user", "content": p["query"]}
        ]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tok(text, return_tensors="pt").to(mdl.device)
        with torch.no_grad():
            out = mdl.generate(**inputs, max_new_tokens=512, temperature=0.0, do_sample=False)
        resp = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        # Parse tool calls
        tc_matches = re.findall(r'<tool_call>\s*(.*?)\s*</tool_call>', resp, re.DOTALL)
        tool_calls = []
        for m in tc_matches:
            try: tool_calls.append(json.loads(m))
            except: pass
        # Score
        if p.get("should_call") is False:
            score = 1.0 if len(tool_calls) == 0 else 0.0
        elif "expected_fn" in p:
            if not tool_calls:
                score = 0.0
            else:
                fn_ok = tool_calls[0].get("name") == p["expected_fn"]
                args_ok = all(
                    str(v).lower() in str(tool_calls[0].get("arguments", {}).get(k, "")).lower()
                    for k, v in p.get("expected_args", {}).items()
                )
                score = (0.5 * fn_ok) + (0.5 * args_ok)
        else:
            score = 0.0
        status = ["FAIL", "WARN", "PASS"][int(score * 2)]
        print(f"    {status} {p['id']}: {score:.0%}")
        scores.append(score)
    del mdl; torch.cuda.empty_cache()
    return np.mean(scores)

def eval_reasoning(model_path, problems):
    """Evaluate code reasoning."""
    print(f"  Loading {model_path.split('/')[-1]}...")
    tok = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    mdl = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True)
    mdl.eval()
    scores = []
    for p in problems:
        msgs = [
            {"role": "system", "content": "Think step-by-step. Show reasoning before code."},
            {"role": "user", "content": p["problem"]}
        ]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = tok(text, return_tensors="pt").to(mdl.device)
        with torch.no_grad():
            out = mdl.generate(**inputs, max_new_tokens=1024, temperature=0.0, do_sample=False)
        resp = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        # Extract code
        code_blocks = re.findall(r'```python\s*(.*?)```', resp, re.DOTALL)
        code = code_blocks[-1] if code_blocks else resp
        # Check reasoning
        has_reason = any(w in resp.lower() for w in ["step", "first", "then", "because", "approach"])
        # Run tests
        passed = 0
        for t in p["tests"]:
            try:
                g = {}; exec(code, g)
                fn = g.get(p["fn_name"])
                if fn and fn(t["input"]) == t["expected"]:
                    passed += 1
            except: pass
        correctness = passed / len(p["tests"]) if p["tests"] else 0
        combined = (0.7 * correctness) + (0.3 * (1.0 if has_reason else 0.0))
        print(f"    {p['id']}: code={correctness:.0%} reason={'Y' if has_reason else 'N'} -> {combined:.0%}")
        scores.append(combined)
    del mdl; torch.cuda.empty_cache()
    return np.mean(scores)

print("Evaluation functions ready")

In [ ]:
# Run all benchmarks
print("=" * 60)
print("BENCHMARKING ALL MODELS")
print("=" * 60)

results = {}
for name, path in EVAL_MODELS.items():
    print(f"\n{'='*40}")
    print(f"Model: {name}")
    print(f"{'='*40}")
    try:
        tc = eval_tool_calling(path, TC_PROBLEMS)
        rs = eval_reasoning(path, REASON_PROBLEMS)
        results[name] = {"tool_calling": tc, "reasoning": rs}
        print(f"  TOOL CALLING: {tc:.0%}")
        print(f"  REASONING:    {rs:.0%}")
    except Exception as e:
        print(f"  ERROR: {e}")

# Summary table
print("\n" + "=" * 60)
print("FINAL RESULTS")
print("=" * 60)
print(f"{'Model':<20} {'Tool Calling':>15} {'Reasoning':>15}")
print("-" * 50)
for name, res in results.items():
    print(f"{name:<20} {res['tool_calling']:>14.0%} {res['reasoning']:>14.0%}")

with open("benchmarks/results.json", "w") as f:
    json.dump({"timestamp": datetime.now().isoformat(), "results": {
        k: {kk: float(vv) for kk, vv in v.items()} for k, v in results.items()
    }}, f, indent=2)
print("\nResults saved to benchmarks/results.json")

---
## Section 6: Export to GGUF for Local Use
> Convert your fine-tuned model to GGUF format for llama.cpp / Ollama.


In [ ]:
# Install llama.cpp conversion tools
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp
!cd /tmp/llama.cpp && pip install -q -r requirements.txt
!cd /tmp/llama.cpp && cmake -B build && cmake --build build --target llama-quantize -j4

# Find the best model to export
EXPORT_MODEL = None
for path in ["outputs/stage3_dpo/merged", "outputs/stage2/merged", "outputs/stage1/merged"]:
    if os.path.exists(path):
        EXPORT_MODEL = path
        break

if EXPORT_MODEL is None:
    print("No fine-tuned model found! Run training sections first.")
else:
    print(f"Exporting: {EXPORT_MODEL}")

In [ ]:
import subprocess

if EXPORT_MODEL:
    OUTPUT_NAME = "code-llm-forge-v1"

    # Step 1: Convert to FP16 GGUF
    print("Converting to GGUF FP16...")
    r = subprocess.run(
        f"python /tmp/llama.cpp/convert_hf_to_gguf.py {EXPORT_MODEL} "
        f"--outfile gguf_output/{OUTPUT_NAME}-f16.gguf --outtype f16",
        shell=True, capture_output=True, text=True)
    if r.returncode == 0:
        print("FP16 GGUF created")
    else:
        print(f"Error: {r.stderr[:300]}")

    # Step 2: Quantize
    QUANTS = ["Q4_K_M", "Q5_K_M", "Q8_0"]
    for q in QUANTS:
        print(f"Quantizing {q}...")
        r = subprocess.run(
            f"/tmp/llama.cpp/build/bin/llama-quantize "
            f"gguf_output/{OUTPUT_NAME}-f16.gguf gguf_output/{OUTPUT_NAME}-{q}.gguf {q}",
            shell=True, capture_output=True, text=True)
        if r.returncode == 0:
            sz = os.path.getsize(f"gguf_output/{OUTPUT_NAME}-{q}.gguf") / (1024**2)
            print(f"  {q}: {sz:.0f} MB")
        else:
            print(f"  Error: {r.stderr[:200]}")

    # Summary
    print("\n" + "=" * 50)
    print("GGUF FILES READY")
    print("=" * 50)
    for f in sorted(os.listdir("gguf_output")):
        if f.endswith(".gguf"):
            sz = os.path.getsize(f"gguf_output/{f}") / (1024**2)
            print(f"  {f}: {sz:,.0f} MB")

---
## Run Your Model Locally

```bash
# Interactive chat
llama-cli -m code-llm-forge-v1-Q8_0.gguf \
  -co -cnv -fa -ngl 99 -c 8192 \
  -p "You are a helpful coding assistant with tool-calling capabilities."

# OpenAI-compatible API server
llama-server -m code-llm-forge-v1-Q8_0.gguf \
  -ngl 99 -c 8192 --port 8080

# Or with Ollama
# Create a Modelfile, then: ollama create code-forge -f Modelfile
```

---

## Done!
You now have a coding LLM fine-tuned for:
- Code generation across multiple languages
- Tool/function calling with structured JSON output
- Chain-of-thought reasoning

All benchmarked honestly with contamination-resistant evaluations.
